<a href="https://colab.research.google.com/github/kevinketerlondon-tech/Data-Analyst-portfolio/blob/main/00a_fetch_mapillary_metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00a: Fetch Mapillary metadata for the annotated images
**Kevin Keter · University of East London**

For every annotated image (IDs read from the Roboflow export file names) this asks the Mapillary API for its GPS
position, capture time, capture sequence, compass heading and camera type. Output: `mapillary_metadata_312.csv`.
Run time about 2 minutes, no GPU needed. You will be asked for your Mapillary token (it is not stored).

## 1. Settings

In [ ]:
import os, glob, zipfile
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Dissertation/revision"
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
class Args: pass
args = Args()
# Roboflow COCO-segmentation export (Version 3, full resolution), unzipped once
zips = [z for z in glob.glob(f"{DRIVE}/*.zip") if "v3" in os.path.basename(z).lower()]
assert zips, "Put the Roboflow v3 COCO-segmentation zip in " + DRIVE
args.coco_dir = f"{WORK}/roboflow_v3"
if not os.path.exists(f"{args.coco_dir}/train"):
    zipfile.ZipFile(zips[0]).extractall(args.coco_dir)
print("Dataset:", args.coco_dir)
# If the file already exists (it was used for the reported results), write a second copy and compare,
# so re-running this notebook never overwrites the original.
MAIN = f"{DRIVE}/mapillary_metadata_312.csv"
args.out = MAIN if not os.path.exists(MAIN) else MAIN.replace(".csv", "_rerun.csv")
print("Output:", args.out)

Mounted at /content/drive
Dataset: /content/work/roboflow_v3
Output: /content/drive/MyDrive/Dissertation/revision/mapillary_metadata_312_rerun.csv


## 2. Run

In [ ]:
import csv, getpass, glob, json, os, time
from datetime import datetime, timezone
import requests

TOKEN = os.environ.get("MAPILLARY_TOKEN") or getpass.getpass("Mapillary token: ")
FIELDS = "id,geometry,computed_geometry,captured_at,sequence,compass_angle,computed_compass_angle,camera_type,make,model,width,height"

images = []
for split in ["train", "valid", "test"]:
    p = os.path.join(args.coco_dir, split, "_annotations.coco.json")
    if os.path.exists(p):
        images += [(im["file_name"].split("_jpg")[0], split) for im in json.load(open(p))["images"]]

rows, failed = [], []
for n, (image_id, split) in enumerate(images, 1):
    try:
        r = requests.get(f"https://graph.mapillary.com/{image_id}", params={"access_token": TOKEN, "fields": FIELDS}, timeout=20)
        if r.status_code != 200:
            failed.append((image_id, r.status_code)); continue
        d = r.json(); g = d.get("computed_geometry") or d.get("geometry") or {}
        lng, lat = (g.get("coordinates") or [None, None])[:2]; ts = d.get("captured_at")
        rows.append(dict(image_id=image_id, original_split=split, lat=lat, lng=lng,
                         captured_at=datetime.fromtimestamp(ts / 1000, tz=timezone.utc).isoformat() if ts else "",
                         sequence=d.get("sequence", ""), compass_angle=d.get("computed_compass_angle", d.get("compass_angle", "")),
                         camera_type=d.get("camera_type", ""), make=d.get("make", ""), model=d.get("model", ""),
                         width=d.get("width", ""), height=d.get("height", "")))
    except requests.RequestException as e:
        failed.append((image_id, str(e)[:80]))
    if n % 50 == 0: print(f"{n}/{len(images)}")
    time.sleep(0.1)
with open(args.out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print(f"Fetched {len(rows)}/{len(images)}; failed: {failed}")

Mapillary token: ··········
50/312
100/312
150/312
200/312
250/312
300/312
Fetched 310/312; failed: [('179333945215504-Copy', 400), ('1041618168551168', 400)]


### Check against the file used in the paper

In [ ]:
if args.out != MAIN:
    a, b = pd.read_csv(MAIN, dtype=str), pd.read_csv(args.out, dtype=str)
    m = a.merge(b, on="image_id", suffixes=("_paper", "_now"))
    moved = (abs(m.lat_paper.astype(float) - m.lat_now.astype(float)) + abs(m.lng_paper.astype(float) - m.lng_now.astype(float)) > 1e-6).sum()
    print(f"Images in paper file: {len(a)} | fetched now: {len(b)} | matched: {len(m)} | positions changed: {moved}")

Images in paper file: 310 | fetched now: 310 | matched: 310 | positions changed: 0
